# 06. Területek Összevetése: Közös Modell, Közös Mértékegység

**Cél:** a regisztrált területek — a fő mintaterület és a kontrollterület — egyetlen közös becslésben történő összehasonlítása a vasúti sávok árazása, az árszint és a mintaösszetétel mentén.

---

### 📖 MIT csinál ez a fejezet?
A két terület garantált pontosságú eladó hirdetéseit egyetlen, közös („pooled”) regresszióba teszi. A fajlagos árakat közös mértékegységre (euró) normalizálja, a fizikai lakásjellemzők és a vasúti sávok hatását együtt becsli, majd két eszközzel méri a területek közötti eltérést: egy terület-mutatóval (árszintkülönbség) és terület × sáv interakciókkal (a sávonkénti árhátrány területek közötti eltérése). Emellett táblázatba foglalja a két terület mintaösszetételét is.

### 📖 MIÉRT van erre szükség?
Két terület árainak nyers összevetése félrevezető lenne: a kínálat összetétele (lakásméret, állapot, sávszerkezet) területenként eltér, a pénznem pedig különböző. Csak egy közös modell tudja elválasztani azt, ami a lakások jellemzőiből fakad, attól, ami valóban az adott terület sajátja.

### 📖 HOGYAN működik?
Minden becslés a központi elemző modul pooled összehasonlító függvényében fut, az eredmények a cache-ből érkeznek — a notebook egyszer fut le, és mindkét terület adatát ugyanaz a függvény tölti be, ezért az aktív terület beállítása ezt a fejezetet nem befolyásolja. A modell méretét, magyarázóerejét és az interakciók számát a visszaadott meta-tábla tartalmazza.

### 📖 Ki a referencia?
A területek szerepe a területi konfigurációból (`data/areas.yaml`) származik: a „fő mintaterület” és a „kontrollterület” elnevezés dinamikus. A regresszióban az árszintkülönbséget a kontrollterület mutatója fejezi ki a fő mintaterülethez képest, a sáv-interakciók pedig azt mutatják, hogy ugyanabban a sávban a kontrollterület árhátránya mennyivel tér el a fő mintaterületétől.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
sys.path.insert(0, '..')
from ingatlan_tdk._utils import *
from ingatlan_tdk import analyses as A
setup_plotly()
df = load_szamitott_master()
p = A.pontos_elado()
AREA = os.environ.get('TDK_ACTIVE_AREA', 'kobanya')

import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from IPython.display import display, HTML

cfg_areas = load_areas_config()
AREAS_CFG = cfg_areas.get('areas', {})
ID2NAME = {aid: am.get('name', aid) for aid, am in AREAS_CFG.items()}
ROLE_NEV = {'primary': 'fő mintaterület', 'control': 'kontrollterület'}

print(f"Aktív terület a futtatáskor: {AREA} — ez a fejezet ettől függetlenül mindkét regisztrált területet elemzi.")

### 1. A pooled kanonikus modell

A közös modell a két terület garantált pontosságú eladó hirdetéseit egyetlen regresszióba teszi. A fajlagos árakat euróra normalizáljuk (a nem eurós terület 400 Ft/EUR feltevéssel — explicit normalizálás, nem piaci árfolyam-becslés). A fizikai jellemzők, a kötöttpálya-elérhetőség és a vasúti sávok együtthatói a két területen közösek; a területek árszintkülönbségét egy terület-mutató, a sávhatások területek közötti eltérését pedig terület × sáv interakciók mérik. A táblában az `ia_...` sorok az interakciók; a számukat a `meta['interakciok']` adja.

In [ ]:
pooled = A.pooled_comparison()
os.environ['TDK_ACTIVE_AREA'] = AREA  # a pooled betöltés belsőleg területet vált — visszaállítjuk az aktív beállítást
coef = pooled['coef']
meta = pooled['meta']

n_pool = int(meta['n'].iloc[0])
r2_pool = float(meta['r2'].iloc[0])
ref_zona = str(meta['ref_zona'].iloc[0])
n_inter = int(meta['interakciok'].iloc[0])

print("A fejezet területfüggetlen: az alábbi regisztrált területeket a pooled függvény együtt tölti be:")
for aid, am in AREAS_CFG.items():
    role = am.get('role', 'control')
    print(f"  • {am.get('name', aid)} — {ROLE_NEV.get(role, role)}")

kpi_cards = [
    ("Pooled minta", f"{n_pool} db", "pontos eladó hirdetés a két területről", "#1e3a8a"),
    ("Modell R²", f"{r2_pool:.3f}", "EUR-ra normalizált közös becslés", "#2563eb"),
    ("Referencia-sáv", ref_zona, "a sáv-dummyk báziskategóriája", "#059669"),
    ("Terület × sáv interakciók", f"{n_inter} db", "a meta['interakciok'] értéke", "#7c3aed"),
]
display(HTML(kpi_grid_html(kpi_cards)))

In [ ]:
def _nev(v):
    if v in ID2NAME:
        role = AREAS_CFG[v].get('role', 'control')
        return ID2NAME[v] + " (" + ROLE_NEV.get(role, role) + ")"
    return v

coef_disp = coef.copy()
coef_disp['nev'] = coef_disp['valtozo'].map(_nev)
coef_out = coef_disp[['nev', 'coef', 'p', 'ci_lo', 'ci_hi']].copy()
coef_out.columns = ['Változó', 'Együttható (log EUR/m²)', 'p-érték', 'CI alsó', 'CI felső']
display(HTML("<b>A pooled kanonikus modell együtthatói (EUR-ban, HC1 robusztus hibák):</b>"
             + coef_out.round(4).to_html(index=False)))

terulet_dummy = [v for v in coef['valtozo'] if v in ID2NAME]
referencia_id = next((aid for aid in AREAS_CFG if aid not in terulet_dummy), None)
if terulet_dummy:
    td = terulet_dummy[0]
    b = float(coef.loc[coef['valtozo'] == td, 'coef'].iloc[0])
    b_p = float(coef.loc[coef['valtozo'] == td, 'p'].iloc[0])
    print(f"A terület-dummy ({_nev(td)}) {(np.exp(b)-1)*100:+.1f}%-os árszintkülönbséget mutat "
          f"(p = {b_p:.3g}) a referencia-területhez ({_nev(referencia_id) if referencia_id else 'a dummy-bázis'}) "
          f"képest, azonos lakásjellemzők és sáv mellett.")

### 2. Terület × sáv interakciók

Az ábrán minden pont egy interakció: a kontrollterület adott sávbeli árhátrányának eltérése a fő mintaterület ugyanazon sávbeli árhátrányától, százalékban, 95%-os konfidencia-intervallummal. Ha az intervallum tartalmazza a nullát, a két terület sávhatása statisztikailag nem különíthető el; a negatív érték mélyebb, a pozitív enyhébb kontrollterületi diszkontot jelez.

In [ ]:
ia = coef[coef['valtozo'].str.startswith('ia_')].copy()
ia['sav'] = ia['valtozo'].str.split('__').str[1]
ia['hatas_pct'] = (np.exp(ia['coef']) - 1) * 100
ia['ci_lo_pct'] = (np.exp(ia['ci_lo']) - 1) * 100
ia['ci_hi_pct'] = (np.exp(ia['ci_hi']) - 1) * 100
ia['terulet'] = ia['valtozo'].str.split('__').str[0].str.replace('ia_', '').map(lambda v: ID2NAME.get(v, v))

fig_ia = px.scatter(
    ia, x='sav', y='hatas_pct',
    error_y=ia['hatas_pct'] - ia['ci_lo_pct'],
    error_y_minus=ia['ci_hi_pct'] - ia['hatas_pct'],
    text=ia['p'].map(lambda v: f"p={v:.3f}"),
    title='Terület × vasúti sáv interakciók (95% CI) — a kontrollterület sávhatás-eltérése',
    labels={'sav': 'Vasúti sáv', 'hatas_pct': 'Eltérés a fő mintaterület sávhatásához képest (%)'},
)
fig_ia.add_hline(y=0, line_dash='dash', line_color='#64748b')
fig_ia.update_layout(height=440)
fig_ia.show()

print(f"Interakciók száma: {len(ia)} (a meta['interakciok'] = {n_inter}).")
if len(ia):
    sig_savok = ia.loc[ia['p'] < 0.05, 'sav'].tolist()
    if sig_savok:
        print(f"5%-os szinten szignifikáns interakció: {', '.join(sig_savok)}.")
    else:
        print("Egyetlen interakció sem szignifikáns 5%-os szinten: a két terület sávonkénti árhátránya "
              "statisztikailag nem különíthető el egymástól.")

### 3. Összetételi különbségek

A regresszió mellett érdemes magát a mintát is összehasonlítani: hány pontos eladó hirdetés van területenként, mekkora a medián fajlagos ár euróban, és hogyan oszlik meg a kínálat a vasúti sávok között. A nyers mediánok NEM ceteris paribus összehasonlíthatók — éppen ezért van szükség az 1. szakasz közös modelljére.

In [ ]:
rows_ossz = []
comp = {}
for aid, am in AREAS_CFG.items():
    d = load_szamitott_master(aid)
    e = d[(d['listing_type'] == 'elado') & (d['minta_garantalt_pontos'] == 1)].copy()
    cur = am.get('currency', 'HUF')
    e['nm_ar_eur_o'] = e['nm_ar_eur'] if cur == 'EUR' else e['nm_ar_huf'] / 400.0
    role = am.get('role', 'control')
    name = am.get('name', aid)
    rows_ossz.append({
        'Terület': name + " (" + ROLE_NEV.get(role, role) + ")",
        'Pontos eladó (db)': len(e),
        'Medián fajlagos ár (EUR/m²)': round(float(e['nm_ar_eur_o'].median()), 0),
    })
    zc = e['vasut_zona'].value_counts(normalize=True) * 100
    comp[name] = {z: float(round(zc.get(z, 0.0), 1)) for z in VASUT_IMMISSZIO_LABELS}

ossz_df = pd.DataFrame(rows_ossz)
display(HTML("<b>Mintaösszetétel a két területen (pontos eladó hirdetések):</b>" + ossz_df.to_html(index=False)))
print("Megjegyzés: a medián fajlagos árak EUR-ban szerepelnek; a nem eurós terület 400 Ft/EUR feltevéssel "
      "van átszámítva — ez explicit normalizálás, nem piaci árfolyam-becslés.")

comp_df = pd.DataFrame(comp).T
comp_df.index.name = 'Terület'
comp_df.columns = [z + " (%)" for z in VASUT_IMMISSZIO_LABELS]
display(HTML("<b>Sávösszetétel (a pontos eladó hirdetések megoszlása, %):</b>" + comp_df.round(1).to_html()))

comp_melt = comp_df.reset_index().melt(id_vars='Terület', var_name='Sav', value_name='arany')
comp_melt['Sav'] = comp_melt['Sav'].str.replace(' (%)', '', regex=False)
fig_comp = px.bar(
    comp_melt, x='Sav', y='arany', color='Terület', barmode='group',
    title='A pontos eladó hirdetések megoszlása a vasúti sávokban a két területen',
    labels={'arany': 'Részarány (%)', 'Sav': 'Vasúti sáv'},
    category_orders={'Sav': VASUT_IMMISSZIO_LABELS},
)
fig_comp.update_layout(height=420)
fig_comp.show()

### 4. Mit mond a két terület együtt

A közös modell lényege a külső érvényesség: ha a vasúti sávok árazása két, eltérő piaci környezetű területen hasonló mintázatot mutat, az eredmény nem egyetlen terület sajátossága. Az alábbi cella a fenti táblák és ábrák számai alapján foglalja össze az objektív eredményeket — minden érték futtatás közben, a cache-ből kerül behelyettesítésre.

In [ ]:
print("ÖSSZEFOGLALÁS — a pooled modell cache-éből:")
print(f"A közös (pooled) modell n = {n_pool} pontos eladó hirdetést használ, R² = {r2_pool:.3f}; "
      f"a sáv-dummyk referencia-kategóriája: {ref_zona}.")
if terulet_dummy:
    print(f"A kontrollterület árszintje {(np.exp(b)-1)*100:+.1f}%-kal tér el a referencia-területtől "
          f"azonos jellemzők és sáv mellett (terület-dummy).")
if len(ia):
    for _, r in ia.iterrows():
        sig = 'szignifikáns' if r['p'] < 0.05 else 'nem szignifikáns'
        print(f"  {r['sav']}: a kontrollterület sávhatás-eltérése {r['hatas_pct']:+.1f}% "
              f"(95% CI: {r['ci_lo_pct']:+.1f}% … {r['ci_hi_pct']:+.1f}%; p = {r['p']:.3f}, {sig}).")
for _, r in ossz_df.iterrows():
    print(f"{r['Terület']}: {r['Pontos eladó (db)']} pontos eladó hirdetés, "
          f"medián fajlagos ár {r['Medián fajlagos ár (EUR/m²)']:.0f} EUR/m².")
near_labs = ['<150 m', '150-300 m']
for area_name, cdict in comp.items():
    near = sum(cdict.get(z, 0.0) for z in near_labs)
    print(f"  {area_name}: a 300 m-en belüli sávokban a pontos eladó kínálat {near:.1f}%-a található.")